# Context Compression

## Learning Objectives
1. Compare extractive vs abstractive compression
2. Implement token importance scoring (LLMLingua)
3. Measure compression ratio vs quality trade-off
4. Deploy compression in RAG pipelines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('Context compression simulation (extractive + abstractive)')

## Level 1: Extractive Compression

In [ ]:
# Level 1: Select important sentences (extractive compression)
# Simple scoring: BM25 relevance to query

def bm25_score(sentence_tokens, query_tokens):
    '''Simple BM25-like scoring.'''
    score = 0
    for token in query_tokens:
        if token in sentence_tokens:
            score += 1 / (1 + len(sentence_tokens))
    return score

# Simulate context (5 sentences)
sentences = [
    [f'sentence_{i}_token_{j}' for j in range(20)] for i in range(5)
]
query = ['key', 'information', 'query_token_x']

# Score each sentence
scores = []
for i, sent in enumerate(sentences):
    score = bm25_score(set(sent), set(query))
    scores.append(score)
    print(f'Sentence {i}: score={score:.3f}')

# Select top sentences (50% compression = keep 2.5 → 3 sentences)
compression_target = 0.5  # Keep 50% of sentences
n_keep = max(1, int(len(sentences) * (1 - compression_target)))
top_indices = np.argsort(scores)[-n_keep:]

selected_sentences = [sentences[i] for i in sorted(top_indices)]

print(f'\nExtractive Compression:')
print(f'  Original: {len(sentences)} sentences, {sum(len(s) for s in sentences)} tokens')
print(f'  Compressed: {len(selected_sentences)} sentences, {sum(len(s) for s in selected_sentences)} tokens')
print(f'  Compression ratio: {sum(len(s) for s in selected_sentences) / sum(len(s) for s in sentences) * 100:.1f}%')
print(f'  Selected sentences: {sorted(top_indices)}')

## Level 2: Token-Level Importance (LLMLingua)

In [ ]:
# Level 2: Token-level compression using reference LM
# Score: how much perplexity increases if token is removed

def estimate_token_importance(context_tokens, reference_probs):
    '''Estimate importance using reference LM perplexity drop.
    
    Δ perplexity = perplexity(context without token) - perplexity(context)
    (Higher drop = more important)
    '''
    
    importance = np.zeros(len(context_tokens))
    
    for i in range(len(context_tokens)):
        # Approximate: p(token | context) from reference model
        token_prob = reference_probs[i] if i < len(reference_probs) else 0.01
        
        # Importance: how much perplexity increases if removed
        # perplexity = -log(p), so removing important tokens increases it
        importance[i] = -np.log(max(token_prob, 1e-5))
    
    return importance

# Simulate context
context_length = 500
context_tokens = [f'tok_{i}' for i in range(context_length)]

# Simulate reference model probabilities (some tokens more important)
base_prob = 0.02  # Average probability
importance_pattern = np.concatenate([
    np.full(100, 0.15),   # Tokens 0-100: rare, important
    np.full(200, 0.02),   # Tokens 100-300: common, less important
    np.full(100, 0.08),   # Tokens 300-400: medium
    np.full(100, 0.03),   # Tokens 400-500: very common
])

importance_scores = estimate_token_importance(context_tokens, importance_pattern)

# Iteratively remove lowest importance tokens until target compression
compression_target = 0.3  # Keep 30% of tokens
target_tokens = int(context_length * compression_target)

# Sort by importance, keep top tokens
top_token_indices = np.argsort(importance_scores)[-target_tokens:]
top_token_indices = np.sort(top_token_indices)

print(f'Token-Level Compression (LLMLingua-style):')
print('-' * 60)
print(f'Context length: {context_length} tokens')
print(f'Compression target: keep {compression_target*100:.0f}%')
print(f'Tokens to keep: {target_tokens}')
print(f'Importance range: [{importance_scores.min():.3f}, {importance_scores.max():.3f}]')
print(f'\nToken importance by region:')
regions = [
    ('Rare tokens (0-100)', importance_scores[:100].mean()),
    ('Common tokens (100-300)', importance_scores[100:300].mean()),
    ('Medium tokens (300-400)', importance_scores[300:400].mean()),
    ('Very common (400-500)', importance_scores[400:500].mean()),
]
for name, avg_imp in regions:
    print(f'  {name}: {avg_imp:.3f}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Importance curve
axes[0].plot(range(context_length), importance_scores, linewidth=1, color='steelblue')
axes[0].fill_between(range(context_length), importance_scores, alpha=0.3, color='steelblue')
axes[0].axhline(np.percentile(importance_scores, compression_target*100), color='red', linestyle='--',
               label=f'Keep threshold (top {compression_target*100:.0f}%)')
axes[0].set_xlabel('Token Position in Context')
axes[0].set_ylabel('Importance Score')
axes[0].set_title('Token Importance Distribution (LLMLingua)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Compression ratio vs quality
compression_ratios = np.linspace(0.1, 1.0, 10)
quality_loss = []  # Estimated accuracy loss

for comp_ratio in compression_ratios:
    # Heuristic: quality loss = comp_ratio^-0.5 (removing tokens hurts more at high compression)
    loss = (1 - comp_ratio) ** 0.6 * 0.3  # Max 30% loss at 10% kept
    quality_loss.append(loss)

axes[1].plot(compression_ratios*100, np.array(quality_loss)*100, marker='o', linewidth=2, markersize=8)
axes[1].fill_between(compression_ratios*100, 0, np.array(quality_loss)*100, alpha=0.3)
axes[1].axvline(30, color='green', linestyle='--', alpha=0.5, label='Recommended (30%)')
axes[1].set_xlabel('Context Preserved (%)')
axes[1].set_ylabel('Accuracy Loss (%)')
axes[1].set_title('Compression-Quality Trade-off')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Real-World Example 1: RAG Pipeline Compression

In [ ]:
# Example 1: Compress retrieved documents in RAG
# Retrieve 5 documents, compress each before passing to LLM

num_docs = 5
doc_lengths = [800, 1200, 950, 700, 1100]  # tokens each
query = 'What is quantum computing?'

print('RAG Pipeline with Context Compression:')
print('='*70)

# Strategy 1: No compression (baseline)
total_tokens_no_comp = sum(doc_lengths)
llm_cost_no_comp = total_tokens_no_comp * 0.0001  # tokens per second

# Strategy 2: Compress each doc to 30%
compression_ratio = 0.30
compressed_lengths = [int(length * compression_ratio) for length in doc_lengths]
total_tokens_comp = sum(compressed_lengths)
llm_cost_comp = total_tokens_comp * 0.0001

# Quality: assume 2% accuracy loss per compression level
quality_loss_pct = (1 - compression_ratio) * 0.05 * 100  # 5% max loss at 100% compression

print(f'\nNocompression Baseline:')
print(f'  Total tokens to LLM: {total_tokens_no_comp}')
print(f'  LLM cost: ${llm_cost_no_comp:.4f}')
print(f'  Latency: {total_tokens_no_comp / 100:.0f}ms (100 tokens/ms)')
print(f'  Quality: baseline')

print(f'\nWith Compression (keep {compression_ratio*100:.0f}%):')
print(f'  Total tokens to LLM: {total_tokens_comp}')
print(f'  LLM cost: ${llm_cost_comp:.4f}')
print(f'  Cost savings: {(1 - llm_cost_comp / llm_cost_no_comp)*100:.1f}%')
print(f'  Latency: {total_tokens_comp / 100:.0f}ms (100 tokens/ms)')
print(f'  Latency savings: {(1 - total_tokens_comp / total_tokens_no_comp)*100:.1f}%')
print(f'  Quality loss: {quality_loss_pct:.1f}%')

print(f'\nDocument-by-document:')
print(f'  Doc | Original | Compressed | Reduction')
print(f'  ----|----------|------------|----------')
for i, (orig, comp) in enumerate(zip(doc_lengths, compressed_lengths)):
    reduction = (1 - comp / orig) * 100
    print(f'  {i+1:1d}   | {orig:8d} | {comp:10d} | {reduction:6.0f}%')

# Breakeven analysis
print(f'\nBreakeven:')
print(f'  Cost saved by compression: ${llm_cost_no_comp - llm_cost_comp:.4f}')
print(f'  If compression takes 5ms overhead: worth it if accuracy loss < {quality_loss_pct:.1f}%')

## Real-World Example 2: Abstractive vs Extractive

In [ ]:
# Example 2: Compare extractive vs abstractive compression
# Extractive: select sentences (fast, exact)
# Abstractive: generate summary (slower, lossy)

document = '''
Quantum computing represents a fundamental shift in how we process information.
Unlike classical computers that use bits as their basic unit, quantum computers use quantum bits or qubits.
Qubits can exist in a superposition of both 0 and 1 simultaneously, allowing quantum computers to explore multiple possibilities at once.
This makes quantum computers exponentially more powerful for certain types of problems.
Current quantum computers are still in their infancy, with challenges in maintaining qubit coherence and reducing error rates.
However, progress is being made rapidly, with companies like IBM, Google, and IonQ leading the way.
The applications of quantum computing span cryptography, drug discovery, optimization, and machine learning.
As the technology matures, quantum computers will likely revolutionize industries and solve previously intractable problems.
'''

sentences = document.split('.')
sentences = [s.strip() for s in sentences if s.strip()]

print('Abstractive vs Extractive Compression:')
print('='*70)

# Extractive: keep 50% of sentences
extractive_ratio = 0.5
n_keep = max(1, int(len(sentences) * extractive_ratio))
extractive_summary = '. '.join(sentences[:n_keep]) + '.'
extractive_tokens = len(extractive_summary.split())

# Abstractive: summary (~40% of original)
abstractive_summary = '''Quantum computers use qubits that can be both 0 and 1 simultaneously,
making them exponentially more powerful than classical computers for certain tasks.
Current quantum computers face challenges in coherence and error rates,
but rapid progress by major tech companies is advancing the field.
Applications include cryptography, drug discovery, optimization, and machine learning.'''

abstractive_tokens = len(abstractive_summary.split())
original_tokens = len(document.split())

print(f'\nOriginal document: {original_tokens} tokens')
print(f'Original: {document[:100]}...')

print(f'\nExtractive compression ({extractive_ratio*100:.0f}%):')
print(f'  Tokens: {extractive_tokens} ({extractive_tokens/original_tokens*100:.1f}% of original)')
print(f'  Method: Select top {n_keep}/{len(sentences)} sentences by BM25')
print(f'  Pros: Fast, exact quotes preserved, no hallucination')
print(f'  Cons: May include redundant information')
print(f'  Summary: {extractive_summary[:80]}...')

print(f'\nAbstractive compression:')
print(f'  Tokens: {abstractive_tokens} ({abstractive_tokens/original_tokens*100:.1f}% of original)')
print(f'  Method: Generate summary with smaller LLM')
print(f'  Pros: High compression ratio, distills key ideas')
print(f'  Cons: May lose specific numbers/dates, risk of hallucination')
print(f'  Summary: {abstractive_summary[:80]}...')

print(f'\nRecommendation:')
print(f'  Factual Q&A (numbers matter): Use extractive')
print(f'  Conversational/summary: Use abstractive')
print(f'  Hybrid: Extractive for facts, abstractive for context')

## Real-World Example 3: Adaptive Compression for Dynamic Context

In [ ]:
# Example 3: Adaptive compression based on remaining context budget

def adaptive_compression(context_budget_tokens=4000, num_documents=5, doc_lengths=None):
    '''Adaptively compress documents to fit token budget.'''
    
    if doc_lengths is None:
        doc_lengths = [800, 1200, 950, 700, 1100]  # Default
    
    total_original = sum(doc_lengths)
    
    if total_original <= context_budget_tokens:
        return doc_lengths, 1.0  # No compression needed
    
    # Binary search for compression ratio
    target_ratio = context_budget_tokens / total_original
    
    # Adaptive: compress harder documents more
    # Importance: inverse of length (longer docs can afford more compression)
    doc_importance = np.array([1.0 / (l / 100) for l in doc_lengths])
    doc_importance /= doc_importance.sum()
    
    compressed_lengths = []
    total_compressed = 0
    
    for orig_len, importance in zip(doc_lengths, doc_importance):
        # Docs with lower importance get compressed more
        # target_ratio ranges from 0.6 (high importance) to 0.3 (low importance)
        doc_target_ratio = 0.3 + (importance * 0.3)
        comp_len = int(orig_len * doc_target_ratio)
        compressed_lengths.append(comp_len)
        total_compressed += comp_len
    
    # Iteratively adjust if still over budget
    while total_compressed > context_budget_tokens and max(compressed_lengths) > 100:
        # Compress the longest document more
        max_idx = np.argmax(compressed_lengths)
        compressed_lengths[max_idx] = int(compressed_lengths[max_idx] * 0.9)
        total_compressed = sum(compressed_lengths)
    
    actual_ratio = total_compressed / total_original
    return compressed_lengths, actual_ratio

# Test with different budgets
budgets = [5000, 4000, 3000, 2000]
doc_lengths_ref = [800, 1200, 950, 700, 1100]
total_orig = sum(doc_lengths_ref)

print('Adaptive Compression for Token Budget:')
print('='*70)
print(f'Document lengths: {doc_lengths_ref}')
print(f'Total original: {total_orig} tokens\n')

for budget in budgets:
    comp_lengths, ratio = adaptive_compression(budget, doc_lengths=doc_lengths_ref)
    total_comp = sum(comp_lengths)
    
    print(f'Budget: {budget} tokens')
    print(f'  Compression ratio: {ratio*100:.1f}%')
    print(f'  Compressed lengths: {comp_lengths}')
    print(f'  Total: {total_comp} (fits budget: {total_comp <= budget})')
    print()

## Comparison: Compression Strategies on Real Workloads

In [ ]:
# Comparison: Which compression strategy wins for different use cases?

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Compression ratio vs speed
methods = ['None', 'Extractive\n(BM25)', 'LLMLingua\n(token-level)', 'Abstractive\n(LLM summary)']
compression_ratios = [1.0, 0.50, 0.35, 0.25]
processing_times = [0, 5, 15, 100]  # ms per document

ax = axes[0, 0]
colors = ['gray', 'steelblue', 'green', 'orange']
bars = ax.barh(methods, processing_times, color=colors, alpha=0.7)
ax2 = ax.twiny()
ax2.barh(methods, compression_ratios, alpha=0.0)
ax.set_xlabel('Processing Time (ms/doc)', color='steelblue')
ax2.set_xlabel('Compression Ratio', color='green')
ax.set_title('Compression Speed Trade-off')
ax.grid(axis='x', alpha=0.3)

# 2. Quality vs compression
compression_levels = np.linspace(0.2, 1.0, 20)
extractive_quality = 100 - (1 - compression_levels) * 8  # 8% loss at 80% compression
llmlingua_quality = 100 - (1 - compression_levels) * 3  # 3% loss at 80% compression
abstractive_quality = 100 - (1 - compression_levels) * 5  # 5% loss

ax = axes[0, 1]
ax.plot(compression_levels * 100, extractive_quality, marker='o', linewidth=2.5, label='Extractive', markersize=6)
ax.plot(compression_levels * 100, llmlingua_quality, marker='s', linewidth=2.5, label='LLMLingua', markersize=6)
ax.plot(compression_levels * 100, abstractive_quality, marker='^', linewidth=2.5, label='Abstractive', markersize=6)
ax.set_xlabel('Context Preserved (%)')
ax.set_ylabel('F1 Score')
ax.set_title('Quality vs Compression Ratio')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim([94, 100.5])

# 3. Cost analysis: compression time vs LLM inference savings
original_contexts = np.array([1000, 2000, 4000, 8000])  # tokens
llm_latency_per_1k = 5  # ms per 1000 tokens

ax = axes[1, 0]
# Assume 40% compression with 15ms overhead
compression_overhead = 15  # ms (fixed)
latency_no_comp = original_contexts / 1000 * llm_latency_per_1k
latency_with_comp = compression_overhead + (original_contexts * 0.6) / 1000 * llm_latency_per_1k
latency_saved = latency_no_comp - latency_with_comp

x = np.arange(len(original_contexts))
width = 0.35
ax.bar(x - width/2, latency_no_comp, width, label='No compression', alpha=0.7)
ax.bar(x + width/2, latency_with_comp, width, label='With compression', alpha=0.7)
ax.set_xlabel('Document Size (tokens)')
ax.set_ylabel('Total Latency (ms)')
ax.set_title('Latency: Compression Overhead vs LLM Savings')
ax.set_xticks(x)
ax.set_xticklabels([f'{int(c)}' for c in original_contexts])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 4. Use case matrix
use_cases = ['Factual QA', 'Chat\nMemory', 'Long Doc\nAnalysis', 'Code\nReview']
extractive_score = [9, 5, 6, 8]
llmlingua_score = [8, 6, 8, 7]
abstractive_score = [6, 9, 7, 4]

x = np.arange(len(use_cases))
width = 0.25

ax = axes[1, 1]
ax.bar(x - width, extractive_score, width, label='Extractive', alpha=0.8)
ax.bar(x, llmlingua_score, width, label='LLMLingua', alpha=0.8)
ax.bar(x + width, abstractive_score, width, label='Abstractive', alpha=0.8)
ax.set_ylabel('Suitability Score (0-10)')
ax.set_title('Best Compression Method by Use Case')
ax.set_xticks(x)
ax.set_xticklabels(use_cases)
ax.legend()
ax.set_ylim([0, 10])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\nRecommendation by Use Case:')
print('='*70)
print('Factual Q&A: Extractive (preserves exact numbers, dates)')
print('Chat Memory: Abstractive (distills conversation essence)')
print('Long Documents: LLMLingua (token importance)')
print('Code Review: Extractive (need exact code snippets)')
print('')
print('Default strategy: Start with Extractive, upgrade to LLMLingua for cost-sensitive')

## Key Takeaways

In [ ]:
print('='*70)
print('CONTEXT COMPRESSION BEST PRACTICES')
print('='*70)
print('')
print('1. EXTRACTIVE COMPRESSION')
print('   - Fast: O(n) with BM25 or cross-encoder scoring')
print('   - Lossless for selected content')
print('   - Cannot synthesize across passages')
print('   - Use for: factual Q&A, retrieval-augmented generation')
print('')
print('2. LLMLingua (TOKEN-LEVEL)')
print('   - Score tokens by perplexity impact')
print('   - Keep top important tokens (not sentences)')
print('   - Achieves 40-50% compression with <2% quality loss')
print('   - Use for: long documents, cost-sensitive applications')
print('')
print('3. ABSTRACTIVE COMPRESSION')
print('   - Generate summary with small LLM')
print('   - Highest information density')
print('   - Risk: loses numbers, dates, specific quotes')
print('   - Use for: conversational context, long histories')
print('')
print('4. COMPRESSION RATIOS')
print('   - 30-50% compression: minimal accuracy loss (<2%)')
print('   - 50-70% compression: some loss (<5%), but still good')
print('   - >70% compression: significant quality drop')
print('')
print('5. PRACTICAL WORKFLOW')
print('   - For RAG: Extract doc chunks → compress → concatenate')
print('   - Compression cost: 5-20ms per document')
print('   - LLM prefill saved: 50-70% (primary benefit)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Query-Aware Compression')
print('-' * 70)
print('Current: Compress context ignoring query')
print('Better: Score tokens/sentences by relevance to query')
print('')
print('Implement: Cross-encoder(query, token_context)')
print('Then select top tokens by relevance to query')
print('Expected: Higher quality at same compression ratio')
print('')
print('Exercise 2: Cost-Benefit Analysis')
print('-' * 70)
print('Compare:')
print('  1. No compression: slow LLM inference')
print('  2. Compression: fast LLM inference + compression overhead')
print('\nWhen does compression break even?')
print('Hint: Breakeven ~ compression_time < (1 - compression_ratio) * llm_time')
print('')
print('Success! Generated notebook 54 (context-compression)')